In [1]:
import os
from os.path import dirname

root_dir = dirname(os.getcwd())
os.chdir(root_dir)

In [2]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.nn import *
from src.utils import *

In [3]:
def trim_to_multiple(arr, multiple):
    length = len(arr)
    trimmed_length = length - (length % multiple)
    return arr[:trimmed_length]

In [4]:
car_df = pd.read_csv('data/car-1/Accelerometer.csv')
car_x = car_df['x'].to_numpy()
car_y = car_df['y'].to_numpy()
car_z = car_df['z'].to_numpy()
car_time = pd.to_datetime(car_df['timestamp'])

In [5]:
bike_df = pd.read_csv('data/bike/Accelerometer.csv')
bike_x = bike_df['x'].to_numpy()
bike_y = bike_df['y'].to_numpy()
bike_z = bike_df['z'].to_numpy()
bike_time = pd.to_datetime(bike_df['timestamp'])

In [6]:
car_time_offset_ms = (car_time - car_time[0]).astype('timedelta64[ms]').to_numpy(dtype=np.float32)
car_time_offset_s = car_time_offset_ms / 1000.0

In [7]:
bike_time_offset_ms = (bike_time - bike_time[0]).astype('timedelta64[ms]').to_numpy(dtype=np.float32)
bike_time_offset_s = bike_time_offset_ms / 1000.0

In [8]:
car_max_time_s = np.floor(np.max(car_time_offset_s))
bike_max_time_s = np.floor(np.max(bike_time_offset_s))

car_total_points = car_max_time_s * SENSOR_RATE
bike_total_points = bike_max_time_s * SENSOR_RATE

In [9]:
car_eval_time = np.linspace(0, car_max_time_s, int(car_total_points))
bike_eval_time = np.linspace(0, bike_max_time_s, int(bike_total_points))

In [10]:
car_x_interp = trim_to_multiple(np.interp(car_eval_time, car_time_offset_s, car_x), 50)
car_y_interp = trim_to_multiple(np.interp(car_eval_time, car_time_offset_s, car_y), 50)
car_z_interp = trim_to_multiple(np.interp(car_eval_time, car_time_offset_s, car_z), 50)

bike_x_interp = trim_to_multiple(np.interp(bike_eval_time, bike_time_offset_s, bike_x), 50)
bike_y_interp = trim_to_multiple(np.interp(bike_eval_time, bike_time_offset_s, bike_y), 50)
bike_z_interp = trim_to_multiple(np.interp(bike_eval_time, bike_time_offset_s, bike_z), 50)

In [11]:
car_stacked_data = torch.stack(
    [
        torch.tensor(car_x_interp, dtype=torch.float32),
        torch.tensor(car_y_interp, dtype=torch.float32),
        torch.tensor(car_z_interp, dtype=torch.float32),
    ],
    dim=1,
)

car_reshaped_data = torch.reshape(car_stacked_data, (-1, WINDOW_SIZE, 3)).transpose(1, 2).contiguous()

In [12]:
bike_stacked_data = torch.stack(
    [
        torch.tensor(bike_x_interp, dtype=torch.float32),
        torch.tensor(bike_y_interp, dtype=torch.float32),
        torch.tensor(bike_z_interp, dtype=torch.float32),
    ],
    dim=1,
)

bike_reshaped_data = torch.reshape(bike_stacked_data, (-1, WINDOW_SIZE, 3)).transpose(1, 2).contiguous()

In [13]:
car_label = torch.full((car_stacked_data.shape[0],), 2, dtype=torch.long)
car_test_data = {
    'data': car_reshaped_data,
    'label': car_label,
}
car_test_dataloader = get_dataloader(car_test_data)

In [14]:
bike_label = torch.full((bike_stacked_data.shape[0],), 0, dtype=torch.long)
bike_test_data = {
    'data': bike_reshaped_data,
    'label': bike_label,
}
bike_test_dataloader = get_dataloader(bike_test_data)

In [15]:
nn_model = NeuralNetwork()
nn_state = torch.load('results/neural_network/averaged_model/best_model.pth')
nn_model.load_state_dict(nn_state)
nn_model.eval()

NeuralNetwork(
  (hidden_activation): ReLU()
  (final_activation): Softmax(dim=-1)
  (fc1): Linear(in_features=150, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)

In [16]:
bike_y_interp

array([-0.704224  , -0.67167609, -0.6388474 , ..., -0.26814553,
       -0.27447711, -0.28078937], shape=(35850,))

In [17]:
cnn_model = ConvolutionalNeuralNetwork()
cnn_state = torch.load('results/convolutional_neural_network/averaged_model/best_model.pth')
cnn_model.load_state_dict(cnn_state)
cnn_model.eval()

ConvolutionalNeuralNetwork(
  (conv1): Conv1d(3, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=384, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=3, bias=True)
  (hidden_activation): ReLU()
  (final_activation): Softmax(dim=-1)
)

In [18]:
nn_true_label = []
nn_pred_label = []

In [19]:
with torch.no_grad():
    for inputs, labels in car_test_dataloader:
        outputs = nn_model(inputs)
        _, predicted = torch.max(outputs, 1)

        nn_true_label.extend(labels.cpu().numpy())
        nn_pred_label.extend(predicted.cpu().numpy())

nn_eval_metrics = calculate_metrics(nn_true_label, nn_pred_label)

In [20]:
nn_eval_metrics

{'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}

In [21]:
cnn_true_label = []
cnn_pred_label = []

In [22]:
with torch.no_grad():
    for inputs, labels in car_test_dataloader:
        outputs = cnn_model(inputs)
        _, predicted = torch.max(outputs, 1)

        cnn_true_label.extend(labels.cpu().numpy())
        cnn_pred_label.extend(predicted.cpu().numpy())

cnn_eval_metrics = calculate_metrics(cnn_true_label, cnn_pred_label)

In [23]:
cnn_eval_metrics

{'accuracy': 0.0015495867768595042,
 'precision': 0.0015495867768595042,
 'recall': 0.0015495867768595042,
 'f1_score': 0.0015495867768595042}

In [24]:
from collections import Counter
Counter(cnn_pred_label)

Counter({np.int64(0): 1933, np.int64(2): 3})

In [25]:
Counter(nn_pred_label)

Counter({np.int64(1): 1936})